In [ ]:
pip install transformers==4.44.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 67.5 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
    Uninstalling transformers-4.57.6:
      Successfully uninstalled transformers-4.57.6


In [ ]:
import torch
import torch.nn as nn
from transformers import GPT2LMHeadModel, GPT2Tokenizer, Trainer, TrainingArguments, default_data_collator
from datasets import load_dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# ================================================
# 1. Dataset準備
# ================================================
def prepare_dataset():
    print("Loading WikiText-2 dataset from raw URLs...")
    data_urls = {
        "train": "https://raw.githubusercontent.com/pytorch/examples/master/word_language_model/data/wikitext-2/train.txt",
        "validation": "https://raw.githubusercontent.com/pytorch/examples/master/word_language_model/data/wikitext-2/valid.txt",
        "test": "https://raw.githubusercontent.com/pytorch/examples/master/word_language_model/data/wikitext-2/test.txt"
    }
    dataset = load_dataset("text", data_files=data_urls)

    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token

    def tokenize_function(examples):
        return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

    tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
    tokenized_datasets = tokenized_datasets.map(
        lambda batch: {"labels": batch["input_ids"]}, batched=True
    )
    tokenized_datasets.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

    return tokenized_datasets["train"], tokenized_datasets["validation"], tokenized_datasets["test"], tokenizer

# ================================================
# 2. BaselineTrainer (AdamW版 - デフォルト)
# ================================================
class AdamBaselineTrainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
        self.global_step_counter = 0
        self.history = []
        self.loss_buffer = []
        self.grad_buffer = []

    # create_optimizer はオーバーライドせず、デフォルトの AdamW を使用します

    def training_step(self, model, inputs):
        model.train()
        inputs = self._prepare_inputs(inputs)

        with self.compute_loss_context_manager():
            loss = self.compute_loss(model, inputs)

        if self.args.n_gpu > 1:
            loss = loss.mean()
        loss.backward()

        # 勾配ノルム記録
        total_norm = 0.0
        for p in model.parameters():
            if p.grad is not None:
                param_norm = p.grad.data.norm(2)
                total_norm += param_norm.item() ** 2
        total_norm = total_norm ** 0.5

        self.grad_buffer.append(total_norm)
        self.loss_buffer.append(loss.item())
        self.global_step_counter += 1

        optimizer = self.optimizer

        # ★公平な比較のため、こちらも MaxGrad=4.0 でクリッピング（ただし通常の勾配は約2.0なので影響なし）
        if self.args.max_grad_norm is not None and self.args.max_grad_norm > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), self.args.max_grad_norm)

        optimizer.step()
        optimizer.zero_grad()

        if self.global_step_counter % 100 == 0:
            avg_train_loss = np.mean(self.loss_buffer)
            avg_grad_norm = np.mean(self.grad_buffer)
            self.loss_buffer = []
            self.grad_buffer = []

            val_metrics = self.evaluate(eval_dataset=self.eval_dataset)
            val_loss = val_metrics['eval_loss']
            self.model.train()

            self.history.append({
                "step": self.global_step_counter,
                "train_loss": avg_train_loss,
                "val_loss": val_loss,
                "grad_norm": avg_grad_norm
            })
            print(f"[Step {self.global_step_counter}] Train Loss: {avg_train_loss:.4f}, Val Loss: {val_loss:.4f}, Grad: {avg_grad_norm:.4f}")

        return loss.detach()

def main():
    train_dataset, eval_dataset, test_dataset, tokenizer = prepare_dataset()
    print("Loading GPT-2 model...")
    model = GPT2LMHeadModel.from_pretrained("gpt2")
    model.resize_token_embeddings(len(tokenizer))

    training_args = TrainingArguments(
        output_dir="./results_baseline_adam_norm4",
        evaluation_strategy="no",
        per_device_train_batch_size=2,
        per_device_eval_batch_size=4,
        learning_rate=5e-5,
        num_train_epochs=1,
        weight_decay=0.01,
        logging_steps=100,
        save_strategy="no",
        report_to="none",
        dataloader_num_workers=2,
        max_grad_norm=4.0  # ★重要: 4.0設定
    )

    trainer = AdamBaselineTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
        data_collator=default_data_collator
    )

    print("🚀 Starting Baseline Training (AdamW, MaxGrad=4.0)...")
    trainer.train()

    print("\n🔍 Evaluating on Test Set...")
    metrics = trainer.evaluate(eval_dataset=test_dataset)
    print(f"✅ Final Test Loss: {metrics['eval_loss']:.4f}")

    df = pd.DataFrame(trainer.history)
    df.to_csv("baseline_adam_norm4_log.csv", index=False)
    model.save_pretrained("./final_gpt2_baseline_adam_norm4")
    print("✅ Completed.")

if __name__ == "__main__":
    main()

Loading WikiText-2 dataset from raw URLs...


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


...

🚀 Starting Baseline Training (AdamW, MaxGrad=4.0)...


Step,Training Loss,Validation Loss
99,No log,1.334901
100,1.771300,No Log
199,1.771300,1.273900
200,1.212100,No Log
299,1.212100,1.261045
300,1.234100,No Log
399,1.234100,1.243596
400,1.420700,No Log
499,1.420700,1.246246
500,1.149700,No Log


[Step 100] Train Loss: 1.7713, Val Loss: 1.3349, Grad: 14.8766
[Step 200] Train Loss: 1.2121, Val Loss: 1.2739, Grad: 6.3440
[Step 300] Train Loss: 1.2341, Val Loss: 1.2610, Grad: 4.9712
[Step 400] Train Loss: 1.4207, Val Loss: 1.2436, Grad: 4.6563
[Step 500] Train Loss: 1.1497, Val Loss: 1.2462, Grad: 3.1662
[Step 600] Train Loss: 1.2821, Val Loss: 1.2278, Grad: 3.1875
[Step 700] Train Loss: 1.2304, Val Loss: 1.2248, Grad: 2.7789
[Step 800] Train Loss: 1.4804, Val Loss: 1.2187, Grad: 2.9588
[Step 900] Train Loss: 1.2318, Val Loss: 1.2219, Grad: 2.4089
[Step 1000] Train Loss: 1.3147, Val Loss: 1.2110, Grad: 2.6764
[Step 1100] Train Loss: 1.4472, Val Loss: 1.2114, Grad: 2.7043
[Step 1200] Train Loss: 1.3108, Val Loss: 1.2103, Grad: 2.5135
[Step 1300] Train Loss: 1.4776, Val Loss: 1.2104, Grad: 2.6944
[Step 1400] Train Loss: 1.1435, Val Loss: 1.2071, Grad: 2.2861
[Step 1500] Train Loss: 1.3537, Val Loss: 1.2005, Grad: 2.4472
[Step 1600] Train Loss: 1.3158, Val Loss: 1.2049, Grad: 2.4932


Step,Training Loss,Validation Loss
99,No log,1.334901
100,1.771300,No Log
199,1.771300,1.273900
200,1.212100,No Log
299,1.212100,1.261045
300,1.234100,No Log
399,1.234100,1.243596
400,1.420700,No Log
499,1.420700,1.246246
500,1.149700,No Log


[Step 10600] Train Loss: 1.0448, Val Loss: 1.1486, Grad: 1.7520
[Step 10700] Train Loss: 1.0802, Val Loss: 1.1489, Grad: 1.8752
[Step 10800] Train Loss: 1.3489, Val Loss: 1.1496, Grad: 2.0918
[Step 10900] Train Loss: 1.1043, Val Loss: 1.1461, Grad: 1.8170
[Step 11000] Train Loss: 1.0226, Val Loss: 1.1463, Grad: 1.6265
[Step 11100] Train Loss: 1.2471, Val Loss: 1.1462, Grad: 1.9238
[Step 11200] Train Loss: 1.1780, Val Loss: 1.1459, Grad: 2.0172
[Step 11300] Train Loss: 1.3020, Val Loss: 1.1465, Grad: 2.0387
[Step 11400] Train Loss: 1.0060, Val Loss: 1.1469, Grad: 1.6390
[Step 11500] Train Loss: 1.1179, Val Loss: 1.1455, Grad: 1.7569
[Step 11600] Train Loss: 1.1474, Val Loss: 1.1455, Grad: 1.9040
[Step 11700] Train Loss: 1.0879, Val Loss: 1.1457, Grad: 1.6916
[Step 11800] Train Loss: 1.0160, Val Loss: 1.1445, Grad: 1.6659
[Step 11900] Train Loss: 1.2515, Val Loss: 1.1446, Grad: 1.8812
[Step 12000] Train Loss: 1.1753, Val Loss: 1.1436, Grad: 1.9147
[Step 12100] Train Loss: 1.2020, Val Los